# Keras Colab Training Launcher

This notebook is the Colab entry point for training. The real training logic lives in `pipeline/02_train.py`, so the notebook stays small and the project remains reusable from scripts, GitHub, and Colab.

Use this notebook for GPU training. Do not run full training locally.

## 1. Enable GPU

In Colab, go to `Runtime > Change runtime type > T4 GPU`, then run the next cell.

In [1]:
import tensorflow as tf

print('TensorFlow:', tf.__version__)
print('GPU devices:', tf.config.list_physical_devices('GPU'))

TensorFlow: 2.20.0
GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 2. Clone The Repo

This checks out the active migration branch.

In [2]:
!git clone https://github.com/SARWAGYASHAH/Anomalous-Sound-Detection-using-Spectrograms.git
%cd Anomalous-Sound-Detection-using-Spectrograms
!git checkout migrate-to-keras
!git pull

Cloning into 'Anomalous-Sound-Detection-using-Spectrograms'...
remote: Enumerating objects: 206, done.
remote: Counting objects: 100% (206/206), done.
remote: Compressing objects: 100% (128/128), done.
remote: Total 206 (delta 78), reused 177 (delta 52), pack-reused 0 (from 0)
Receiving objects: 100% (206/206), 82.94 KiB | 5.92 MiB/s, done.
Resolving deltas: 100% (78/78), done.
/content/Anomalous-Sound-Detection-using-Spectrograms
Branch 'migrate-to-keras' set up to track remote branch 'migrate-to-keras' from 'origin'.
Switched to a new branch 'migrate-to-keras'
Already up to date.


## 3. Install Dependencies

In [3]:
# Keep Colab's preinstalled TensorFlow/GPU stack intact.
!pip install -q -r requirements-colab.txt
!pip install -q -e . --no-deps

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 115.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 128.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 69.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 887.7/887.7 kB 67.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## 4. Attach Data

The raw gearbox zip is stored on Google Drive and downloaded with `gdown`.

If the download cell fails, open the Drive file sharing settings and set access to `Anyone with the link`.

In [4]:
from pathlib import Path

DRIVE_ZIP_URL = 'https://drive.google.com/file/d/1p6TDo1GpTWHQzfHRcxs7NgXtG4-xVzqS/view?usp=drive_link'
ZIP_PATH = Path('Data/dev_data_gearbox.zip')
ZIP_PATH.parent.mkdir(parents=True, exist_ok=True)

if not ZIP_PATH.exists():
    !gdown --fuzzy "$DRIVE_ZIP_URL" -O Data/dev_data_gearbox.zip

print('Zip exists:', ZIP_PATH.exists())
print('Zip size MB:', round(ZIP_PATH.stat().st_size / (1024 * 1024), 2) if ZIP_PATH.exists() else 'missing')

Downloading...
From (original): https://drive.google.com/uc?id=1p6TDo1GpTWHQzfHRcxs7NgXtG4-xVzqS
From (redirected): https://drive.google.com/uc?id=1p6TDo1GpTWHQzfHRcxs7NgXtG4-xVzqS&confirm=t&uuid=da6329e1-797e-4211-85c4-c0e9f3fa6a22
To: /content/Anomalous-Sound-Detection-using-Spectrograms/Data/dev_data_gearbox.zip
100% 1.20G/1.20G [00:16<00:00, 73.1MB/s]
Zip exists: True
Zip size MB: 1144.64


In [5]:
!unzip -q -o Data/dev_data_gearbox.zip -d Data/
!ls Data
!python pipeline/01_preprocess.py

dev_data_gearbox.zip  gearbox
2026-05-12 06:48:21 | INFO     | src.utils.seed | Random seed set to 42 (random, numpy, tensorflow)
2026-05-12 06:48:21 | INFO     | preprocess | Log file: artifacts/logs/run_20260512_064821.log
2026-05-12 06:48:21 | INFO     | preprocess | ============================================================
2026-05-12 06:48:21 | INFO     | preprocess | PIPELINE STAGE 1: Preprocessing (Audio → Spectrograms)
2026-05-12 06:48:21 | INFO     | preprocess | ============================================================
2026-05-12 06:48:21 | INFO     | preprocess | Computing global spectrogram normalization stats from train/normal only
2026-05-12 06:48:21 | INFO     | src.data.audio_loader | Discovered 3026 .wav files in Data/gearbox/train
2026-05-12 06:48:38 | INFO     | preprocess |   Stats pass processed 200/3026 normal training files
2026-05-12 06:48:40 | INFO     | preprocess |   Stats pass processed 400/3026 normal training files
2026-05-12 06:48:42 | INFO     | pre

## 5. Verify Processed Data

In [6]:
from pathlib import Path

for path in [
    Path('Data/processed/train/normal'),
    Path('Data/processed/source_test/normal'),
    Path('Data/processed/source_test/anomaly'),
    Path('Data/processed/target_test/normal'),
    Path('Data/processed/target_test/anomaly'),
]:
    count = len(list(path.glob('*.npy'))) if path.exists() else 0
    print(path, count)

Data/processed/train/normal 3026
Data/processed/source_test/normal 411
Data/processed/source_test/anomaly 351
Data/processed/target_test/normal 309
Data/processed/target_test/anomaly 336


## 6. Dry Run

This builds the dataset/model and runs one forward pass. It does not train.

In [ ]:
!python pipeline/02_train.py --dry-run --no-mlflow --batch-size 4

2026-05-06 17:53:32 | INFO     | src.utils.seed | Random seed set to 42 (random, numpy, tensorflow)
2026-05-06 17:53:32 | INFO     | train | Log file: artifacts/logs/run_20260506_175332.log
2026-05-06 17:53:32 | INFO     | train | ============================================================
2026-05-06 17:53:32 | INFO     | train | PIPELINE STAGE 2: Training (Keras Autoencoder)
2026-05-06 17:53:32 | INFO     | train | ============================================================
2026-05-06 17:53:32 | INFO     | train | TensorFlow version: 2.20.0
2026-05-06 17:53:32 | INFO     | train | GPU devices: ['/physical_device:GPU:0']
2026-05-06 17:53:32 | INFO     | src.data.dataset | Discovered 3026 spectrograms in Data/processed/train (normal=3026, anomaly=0)
2026-05-06 17:53:32 | INFO     | train | Training data directory: Data/processed/train
2026-05-06 17:53:32 | INFO     | train | Input shape: (128, 313, 1)
2026-05-06 17:53:32 | INFO     | train | Batch size: 4
2026-05-06 17:53:32 | INFO   

## 7. First Short Training Run

Start with a short run to verify Colab paths, GPU, model saving, metadata, and callbacks.

In [ ]:
!python pipeline/02_train.py --config config/default.yaml --epochs 3 --no-mlflow

2026-05-06 17:53:50 | INFO     | src.utils.seed | Random seed set to 42 (random, numpy, tensorflow)
2026-05-06 17:53:50 | INFO     | train | Log file: artifacts/logs/run_20260506_175350.log
2026-05-06 17:53:50 | INFO     | train | ============================================================
2026-05-06 17:53:50 | INFO     | train | PIPELINE STAGE 2: Training (Keras Autoencoder)
2026-05-06 17:53:50 | INFO     | train | ============================================================
2026-05-06 17:53:50 | INFO     | train | TensorFlow version: 2.20.0
2026-05-06 17:53:50 | INFO     | train | GPU devices: ['/physical_device:GPU:0']
2026-05-06 17:53:50 | INFO     | src.data.dataset | Discovered 3026 spectrograms in Data/processed/train (normal=3026, anomaly=0)
2026-05-06 17:53:50 | INFO     | train | Training data directory: Data/processed/train
2026-05-06 17:53:50 | INFO     | train | Input shape: (128, 313, 1)
2026-05-06 17:53:50 | INFO     | train | Batch size: 32
2026-05-06 17:53:50 | INFO  

## 8. Full Training Run

Run this after the short run succeeds. MLflow is enabled by config unless you pass `--no-mlflow`.

In [7]:
!python pipeline/02_train.py --config config/default.yaml

2026-05-12 06:51:05 | INFO     | src.utils.seed | Random seed set to 42 (random, numpy, tensorflow)
2026-05-12 06:51:05 | INFO     | train | Log file: artifacts/logs/run_20260512_065105.log
2026-05-12 06:51:05 | INFO     | train | ============================================================
2026-05-12 06:51:05 | INFO     | train | PIPELINE STAGE 2: Training (Keras Autoencoder)
2026-05-12 06:51:05 | INFO     | train | ============================================================
2026-05-12 06:51:06 | INFO     | train | TensorFlow version: 2.20.0
2026-05-12 06:51:06 | INFO     | train | GPU devices: ['/physical_device:GPU:0']
2026-05-12 06:51:06 | INFO     | src.data.dataset | Discovered 3026 spectrograms in Data/processed/train (normal=3026, anomaly=0)
2026-05-12 06:51:06 | INFO     | train | Training data directory: Data/processed/train
2026-05-12 06:51:06 | INFO     | train | Input shape: (128, 313, 1)
2026-05-12 06:51:06 | INFO     | train | Batch size: 32
2026-05-12 06:51:06 | INFO  

## 9. Inspect Artifacts

In [8]:
!find artifacts -maxdepth 3 -type f | sort | head -80

artifacts/logs/run_20260512_064821.log
artifacts/logs/run_20260512_065105.log
artifacts/metadata/run_20260512_065108.json
artifacts/models/v1/best_model.keras
artifacts/models/v1/config_snapshot.yaml
artifacts/models/v1/final_model.keras
artifacts/models/v1/training_log.csv
artifacts/preprocessing/spectrogram_stats.json


In [9]:
from google.colab import drive
drive.mount("/content/drive")

!mkdir -p "/content/drive/MyDrive/anomalous_sound_detection/artifacts"
!cp -r artifacts/* "/content/drive/MyDrive/anomalous_sound_detection/artifacts/"


Mounted at /content/drive
